# 🤖 Notebook 03 — Modèles Baseline & Comparaison Stratégies Anti-Déséquilibre
**Projet : SentimentsLoupe — Phase 1 Flipkart**

---
## Objectifs
1. Démontrer l'impact du déséquilibre sur un modèle naïf (sans traitement)
2. Comparer 5 stratégies : none / class_weight / random_over / SMOTE / SMOTETomek
3. Évaluer 4 classifieurs : LR / SVM / NaiveBayes / SVM-calibré
4. Identifier le champion (F1-macro)
5. Valider avec cross-validation 5-fold

**Hypothèse** : Sans traitement, le modèle prédit majoritairement *positive*  
→ accuracy élevée (~81%) mais F1-macro faible (~0.33)  
→ F1 sur classe *negative* quasi nul (~0.05)

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.config.settings import MODELS_DIR, PLOTS_DIR, CLASS_NAMES, RANDOM_STATE
from src.models.imbalance_strategies import ImbalanceExperiment

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = {'positive': '#1D9E75', 'neutral': '#888780', 'negative': '#E24B4A'}

print('✅ Setup OK')

## 1. Chargement des Features (depuis Notebook 02)

In [ ]:
data = joblib.load(str(MODELS_DIR / 'features_ready.pkl'))
X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train distribution : {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'y_test  distribution : {dict(zip(*np.unique(y_test,  return_counts=True)))}')

# Label map
ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}

## 2. Démonstration : Pourquoi le Déséquilibre est Dangereux

In [ ]:
# Modèle "idiot" : prédit toujours la classe majoritaire (positive=2)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print('=' * 60)
print('  MODÈLE IDIOT (prédit toujours "positive")')
print('=' * 60)
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_dummy):.4f}  ← semble bien !')
print(f'  F1-Macro  : {f1_score(y_test, y_pred_dummy, average="macro", zero_division=0):.4f}  ← catastrophique')
print()
print(classification_report(y_test, y_pred_dummy,
                              target_names=CLASS_NAMES, zero_division=0))
print('\n⚠️  CONCLUSION: Accuracy = métrique TROMPEUSE sur données déséquilibrées')
print('    Utiliser F1-Macro comme métrique principale !')

## 3. Expérience Complète : 5 Stratégies × 4 Classifieurs

In [ ]:
print('🔬 Lancement de l\'expérience...')
print('   5 stratégies × 4 classifieurs = 20 combinaisons')
print('   ⏱  Durée estimée : 5-15 min selon CPU\n')

experiment = ImbalanceExperiment()
results_df = experiment.run(
    X_train, y_train,
    X_test,  y_test,
    strategies=['none', 'class_weight', 'random_over', 'smote', 'smotetomek']
)

print('\n✅ Expérience terminée !')

In [ ]:
# Tableau complet
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
print('\n📊 RÉSULTATS COMPLETS (triés par F1-Macro) :')
results_df

## 4. Visualisation Comparative

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(
    'Comparaison des Stratégies Anti-Déséquilibre\n'
    'Dataset Flipkart — DarijaLens Phase 1',
    fontsize=14, fontweight='bold'
)

# ── 1. F1-Macro par stratégie × modèle ───────────────────────────────────
pivot_macro = results_df.pivot(index='strategy', columns='model', values='f1_macro')
pivot_macro.plot(kind='bar', ax=axes[0, 0], width=0.75,
                 colormap='tab10', edgecolor='white', linewidth=0.5)
axes[0, 0].set_title('F1-Macro par stratégie', fontweight='bold')
axes[0, 0].set_xlabel('Stratégie')
axes[0, 0].set_ylabel('F1-Macro')
axes[0, 0].axhline(y=0.70, color='red', linestyle='--', alpha=0.5, linewidth=1.5, label='Seuil 0.70')
axes[0, 0].legend(fontsize=8, loc='lower right')
axes[0, 0].tick_params(axis='x', rotation=25)
axes[0, 0].set_ylim(0, 1.0)

# ── 2. F1 sur classe negative (minoritaire) ───────────────────────────────
pivot_neg = results_df.pivot(index='strategy', columns='model', values='f1_negative')
pivot_neg.plot(kind='bar', ax=axes[0, 1], width=0.75,
               colormap='tab10', edgecolor='white', linewidth=0.5)
axes[0, 1].set_title('F1 Classe "negative" (minoritaire)', fontweight='bold',
                      color='#E24B4A')
axes[0, 1].set_xlabel('Stratégie')
axes[0, 1].set_ylabel('F1 Negative')
axes[0, 1].tick_params(axis='x', rotation=25)
axes[0, 1].legend(fontsize=8)
axes[0, 1].set_ylim(0, 1.0)

# ── 3. Heatmap F1-Macro : stratégie × modèle ────────────────────────────
heatmap_data = results_df.pivot_table(values='f1_macro', index='strategy', columns='model')
sns.heatmap(
    heatmap_data,
    ax=axes[1, 0],
    annot=True, fmt='.3f',
    cmap='RdYlGn', vmin=0.3, vmax=0.9,
    linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.8}
)
axes[1, 0].set_title('Heatmap F1-Macro : stratégie × modèle', fontweight='bold')
axes[1, 0].set_xlabel('Modèle')
axes[1, 0].set_ylabel('Stratégie')

# ── 4. Radar : métriques du champion ─────────────────────────────────────
# Trouver le champion
best_row = results_df.loc[results_df['f1_macro'].idxmax()]
metrics = ['f1_macro', 'f1_negative', 'f1_neutral', 'f1_positive', 'accuracy']
labels  = ['F1-Macro', 'F1-Neg', 'F1-Neu', 'F1-Pos', 'Accuracy']
values  = [best_row[m] for m in metrics]
values_dummy = [f1_score(y_test, y_pred_dummy, average='macro', zero_division=0),
                0.0, 0.0, 
                f1_score(y_test, y_pred_dummy, average=None, zero_division=0)[-1],
                accuracy_score(y_test, y_pred_dummy)]

x = np.arange(len(labels))
width = 0.35
bars1 = axes[1, 1].bar(x - width/2, values_dummy, width, label='Sans traitement (Dummy)',
                        color='#D3D1C7', edgecolor='white')
bars2 = axes[1, 1].bar(x + width/2, values, width,
                        label=f'Champion: [{best_row["strategy"]}] {best_row["model"][:15]}',
                        color='#1D9E75', edgecolor='white')
axes[1, 1].set_title('Champion vs Sans Traitement', fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(labels)
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_ylim(0, 1.1)
axes[1, 1].legend(fontsize=9)
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, h + 0.01,
                        f'{h:.3f}', ha='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '03a_strategy_comparison.png'), bbox_inches='tight', dpi=150)
plt.show()

## 5. Champion & Matrice de Confusion

In [ ]:
best_strategy, best_model_name, champion_obj = experiment.get_champion()

print('\n' + '★' * 55)
print(f'  🏆 CHAMPION : [{best_strategy}] {best_model_name}')
print('★' * 55)
m = champion_obj['metrics']
print(f'  F1-Macro         : {m["f1_macro"]:.4f}')
print(f'  F1 Negative      : {m["f1_negative"]:.4f}  ← classe minoritaire')
print(f'  F1 Neutral       : {m["f1_neutral"]:.4f}')
print(f'  F1 Positive      : {m["f1_positive"]:.4f}')
print(f'  Accuracy         : {m["accuracy"]:.4f}')
print(f'  Precision-Macro  : {m["precision_macro"]:.4f}')
print(f'  Recall-Macro     : {m["recall_macro"]:.4f}')
print()
print('Classification Report:')
print(m['classification_report'])

In [ ]:
# Matrice de confusion
import seaborn as sns
import numpy as np

cm = np.array(m['confusion_matrix'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Matrice de Confusion — Champion : [{best_strategy}] {best_model_name}',
             fontsize=12, fontweight='bold')

# Valeurs absolues
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5, linecolor='white')
axes[0].set_xlabel('Prédit', fontsize=11)
axes[0].set_ylabel('Réel', fontsize=11)
axes[0].set_title('Valeurs absolues', fontweight='bold')

# Normalisée (taux)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1], linewidths=0.5, linecolor='white',
            vmin=0, vmax=1)
axes[1].set_xlabel('Prédit', fontsize=11)
axes[1].set_ylabel('Réel', fontsize=11)
axes[1].set_title('Normalisée (taux par classe réelle)', fontweight='bold')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '03b_confusion_matrix.png'), bbox_inches='tight', dpi=150)
plt.show()

# Analyse de la matrice
print(f'\n📊 Analyse de la matrice de confusion:')
for i, name in enumerate(CLASS_NAMES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0
    print(f'  {name:10} → Recall={recall:.3f} | Precision={prec:.3f} | TP={tp} | FN={fn} | FP={fp}')

## 6. Cross-Validation 5-Fold du Champion

In [ ]:
print('⏱  Cross-validation 5-fold...')
cv_summary = experiment.cross_validate_champion(X_train, y_train, cv=5)

print(f'\n📊 Résultats CV 5-fold du Champion:')
print(f'  F1-Macro mean   : {cv_summary["f1_macro_mean"]:.4f}')
print(f'  F1-Macro std    : {cv_summary["f1_macro_std"]:.4f}')
print(f'  F1-Weighted     : {cv_summary["f1_weighted_mean"]:.4f}')
print(f'  Accuracy        : {cv_summary["accuracy_mean"]:.4f}')
print(f'  Scores par fold : {[f"{s:.4f}" for s in cv_summary["cv_scores"]]}')

# Visualiser la variance des scores
fig, ax = plt.subplots(figsize=(8, 4))
folds  = [f'Fold {i+1}' for i in range(len(cv_summary['cv_scores']))]
scores = cv_summary['cv_scores']
bars = ax.bar(folds, scores, color='#3B8BD4', edgecolor='white', linewidth=1)
ax.axhline(y=cv_summary['f1_macro_mean'], color='#E24B4A', linestyle='--',
           linewidth=2, label=f'Moyenne = {cv_summary["f1_macro_mean"]:.4f}')
ax.fill_between(
    range(len(folds)),
    cv_summary['f1_macro_mean'] - cv_summary['f1_macro_std'],
    cv_summary['f1_macro_mean'] + cv_summary['f1_macro_std'],
    alpha=0.15, color='#E24B4A', label=f'± {cv_summary["f1_macro_std"]:.4f} (std)'
)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, score + 0.002,
            f'{score:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title(f'Cross-Validation 5-Fold — Champion [{best_strategy}] {best_model_name}',
             fontweight='bold')
ax.set_ylabel('F1-Macro')
ax.set_ylim(min(scores) * 0.95, max(scores) * 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '03c_cross_validation.png'), bbox_inches='tight', dpi=150)
plt.show()

# Faible variance = modèle stable
if cv_summary['f1_macro_std'] < 0.02:
    print(f'\n✅ Variance faible ({cv_summary["f1_macro_std"]:.4f}) → Modèle STABLE')
else:
    print(f'\n⚠️  Variance élevée ({cv_summary["f1_macro_std"]:.4f}) → Vérifier overfitting')

## 7. Sauvegarde des Modèles

In [ ]:
import json, os
os.makedirs(str(MODELS_DIR), exist_ok=True)

# Tous les modèles
experiment.save(MODELS_DIR)

# Champion
from src.config.settings import CHAMPION_MODEL_PATH
joblib.dump(champion_obj, str(CHAMPION_MODEL_PATH))
print(f'✅ Champion sauvegardé : {CHAMPION_MODEL_PATH}')

# Métriques JSON
metrics_export = {
    'phase': 1,
    'dataset': 'flipkart',
    'best_strategy': best_strategy,
    'best_model': best_model_name,
    'champion_metrics': {
        k: v for k, v in m.items()
        if k not in ('classification_report', 'confusion_matrix')
    },
    'cross_validation': cv_summary,
    'dummy_baseline': {
        'accuracy': float(accuracy_score(y_test, y_pred_dummy)),
        'f1_macro': float(f1_score(y_test, y_pred_dummy, average='macro', zero_division=0))
    },
    'all_results': results_df.to_dict(orient='records')
}

metrics_path = MODELS_DIR / 'phase1_metrics.json'
with open(str(metrics_path), 'w', encoding='utf-8') as f:
    json.dump(metrics_export, f, indent=2, ensure_ascii=False, default=str)
print(f'✅ Métriques JSON : {metrics_path}')

# Export CSV résultats
results_df.to_csv(str(MODELS_DIR / 'experiment_results.csv'), index=False)
print(f'✅ Résultats CSV  : {MODELS_DIR}/experiment_results.csv')

print('\n➡️  Suite : Notebook 04 — Analyse des Résultats & Rapport')